In [1]:
print("Downloading Dataset from Kaggle...")
!kaggle datasets download -d matthewjansen/ucf101-action-recognition
print("Unzipping Dataset...")
!unzip -q ucf101-action-recognition.zip -d dataset_root
print("Dataset ready!")

Dataset URL: https://www.kaggle.com/datasets/matthewjansen/ucf101-action-recognition
License(s): CC0-1.0
100% 6.53G/6.53G [00:47<00:00, 148MB/s]

Unzipping Dataset...
Dataset ready!


In [6]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import json

# ==========================================
# 1. HYPERPARAMETERS & DIRECTORIES
# ==========================================
DATASET_ROOT = "dataset_root"
TRAIN_CSV = os.path.join(DATASET_ROOT, "train.csv")
VAL_CSV = os.path.join(DATASET_ROOT, "val.csv")
SAVE_MODEL_PATH = "cnn_lstm_action_model.pth"

SEQUENCE_LENGTH = 20
IMAGE_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. BULLETPROOF DATASET EXTRACTION
# ==========================================
print("\nScanning disk for video files...")

# Index all actual videos on disk, completely ignoring file extensions
actual_files_on_disk = {}
for root, dirs, files in os.walk(DATASET_ROOT):
    for file in files:
        if file.endswith(('.avi', '.mp4', '.mkv')):
            base_no_ext = os.path.splitext(file)[0]
            actual_files_on_disk[base_no_ext] = os.path.join(root, file)

def parse_csv_safely(csv_path):
    """Fuzzy matches CSV filenames to actual files on disk."""
    df = pd.read_csv(csv_path)
    file_col = df.columns[0]
    label_col = df.columns[-1]

    valid_paths = []
    valid_labels = []

    for _, row in df.iterrows():
        csv_file_name = str(row[file_col])
        base_name_no_ext = os.path.splitext(os.path.basename(csv_file_name))[0]

        if base_name_no_ext in actual_files_on_disk:
            valid_paths.append(actual_files_on_disk[base_name_no_ext])
            valid_labels.append(row[label_col])

    return pd.DataFrame({'path': valid_paths, 'label_name': valid_labels})

print("Matching CSV entries to actual files...")
train_df = parse_csv_safely(TRAIN_CSV)
val_df = parse_csv_safely(VAL_CSV)

if len(train_df) == 0:
    raise ValueError("CRITICAL ERROR: The CSV filenames are completely different from the downloaded videos.")

# Automatically pick the 3 classes with the most matching videos for fast training
top_3_classes = train_df['label_name'].value_counts().nlargest(3).index.tolist()
print(f"✅ Automatically selected the 3 best classes: {top_3_classes}")

# Filter dataframes to just these 3 classes
train_df = train_df[train_df['label_name'].isin(top_3_classes)]
val_df = val_df[val_df['label_name'].isin(top_3_classes)]

# Map string labels to integers (0, 1, 2)
class_to_idx = {class_name: idx for idx, class_name in enumerate(top_3_classes)}

X_train_paths = train_df['path'].tolist()
y_train = train_df['label_name'].map(class_to_idx).tolist()

X_val_paths = val_df['path'].tolist()
y_val = val_df['label_name'].map(class_to_idx).tolist()

print(f"Final usable data -> Training: {len(X_train_paths)} | Validation: {len(X_val_paths)}\n")

# ==========================================
# 3. PYTORCH DATASET LOADER
# ==========================================
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, sequence_length, image_size):
        self.video_paths = video_paths
        self.labels = labels
        self.sequence_length = sequence_length
        self.image_size = image_size

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((self.image_size, self.image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = []
        cap = cv2.VideoCapture(video_path)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        skip_interval = max(int(frame_count / self.sequence_length), 1)

        for i in range(self.sequence_length):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * skip_interval)
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame_tensor = self.transform(frame)
            frames.append(frame_tensor)

        cap.release()

        while len(frames) < self.sequence_length:
            frames.append(torch.zeros((3, self.image_size, self.image_size)))

        return torch.stack(frames), torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(VideoDataset(X_train_paths, y_train, SEQUENCE_LENGTH, IMAGE_SIZE), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(VideoDataset(X_val_paths, y_val, SEQUENCE_LENGTH, IMAGE_SIZE), batch_size=BATCH_SIZE, shuffle=False)

# ==========================================
# 4. CNN-LSTM MODEL ARCHITECTURE
# ==========================================
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super(CNN_LSTM, self).__init__()

        # Frozen MobileNetV2 for speed (Using modern Weights API)
        mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.feature_extractor = mobilenet.features
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.lstm = nn.LSTM(input_size=1280, hidden_size=64, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        b, seq, c, h, w = x.size()
        x = x.view(b * seq, c, h, w)

        with torch.no_grad():
            x = self.feature_extractor(x)
            x = self.pool(x)

        x = x.view(b * seq, -1)
        x = x.view(b, seq, -1)

        out, _ = self.lstm(x)
        last_out = out[:, -1, :]
        last_out = self.dropout(last_out)
        return self.fc(last_out)

model = CNN_LSTM(num_classes=len(top_3_classes)).to(device)

# --- PRINT MODEL ARCHITECTURE & INFO ---
print("\n" + "="*50)
print("🏛️  MODEL ARCHITECTURE")
print("="*50)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "="*50)
print("📊 MODEL INFORMATION")
print("="*50)
print(f"Total Parameters:      {total_params:,}")
print(f"Frozen Parameters:     {total_params - trainable_params:,} (MobileNetV2)")
print(f"Trainable Parameters:  {trainable_params:,} (LSTM + Fully Connected)")
print(f"Input Shape per Batch: (Batch_Size, {SEQUENCE_LENGTH} frames, 3 channels, {IMAGE_SIZE}x{IMAGE_SIZE})")
print("="*50 + "\n")


# ==========================================
# 5. FAST TRAINING LOOP
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Starting Fast Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_acc = 100. * correct / total
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")

# ==========================================
# 6. EVALUATION
# ==========================================
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, targets in val_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

final_val_acc = 100. * correct / total
print("\n" + "="*50)
print(f"🏆 FINAL VALIDATION ACCURACY: {final_val_acc:.2f}%")
print("="*50)

# ==========================================
# 7. SAVE THE MODEL
# ==========================================
checkpoint = {
    'state_dict': model.state_dict(),
    'classes': top_3_classes
}
torch.save(checkpoint, SAVE_MODEL_PATH)
print(f"✅ Model and classes saved successfully to {SAVE_MODEL_PATH}\n")

# ==========================================
# 8. INFERENCE LOGIC (HOW TO USE IT)
# ==========================================
def predict_video(video_path, saved_model_path=SAVE_MODEL_PATH):
    """Loads the saved weights and predicts a new video."""
    print(f"\nAnalyzing Video: {video_path}...")

    # 1. Load the checkpoint
    chkpt = torch.load(saved_model_path, map_location=device)
    saved_classes = chkpt['classes']

    # 2. Re-initialize model and load weights
    inference_model = CNN_LSTM(num_classes=len(saved_classes)).to(device)
    inference_model.load_state_dict(chkpt['state_dict'])
    inference_model.eval()

    # 3. Process video
    dummy_dataset = VideoDataset([video_path], [0], SEQUENCE_LENGTH, IMAGE_SIZE)
    frames_tensor, _ = dummy_dataset[0]
    input_tensor = frames_tensor.unsqueeze(0).to(device)

    # 4. Predict
    with torch.no_grad():
        outputs = inference_model(input_tensor)
        probabilities = torch.softmax(outputs, dim=1)[0]

    pred_idx = torch.argmax(probabilities).item()
    pred_class = saved_classes[pred_idx]
    confidence = probabilities[pred_idx].item() * 100

    # --- PRINT FINAL PREDICTION RESULT ---
    print("\n" + "="*50)
    print("🎯 FINAL PREDICTION RESULT")
    print("="*50)
    print(f"▶ File:            {video_path}")
    print(f"▶ Detected Action: {pred_class}")
    print(f"▶ Confidence:      {confidence:.2f}%")
    print("="*50 + "\n")

# Example Usage: (Uncomment to test on your own file)
# predict_video("path_to_your_test_video.mp4")

Using device: cuda

Scanning disk for video files...
Matching CSV entries to actual files...
✅ Automatically selected the 3 best classes: ['Basketball', 'CricketShot', 'TennisSwing']
Final usable data -> Training: 447 | Validation: 75

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 150MB/s]



🏛️  MODEL ARCHITECTURE
CNN_LSTM(
  (feature_extractor): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
     

*** Using device: cuda

Scanning disk for video files...
Matching CSV entries to actual files...
✅ Automatically selected the 3 best classes: ['Basketball', 'CricketShot', 'TennisSwing']
Final usable data -> Training: 447 | Validation: 75

Starting Fast Training...
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/10 | Loss: 0.8822 | Train Acc: 59.73%
Epoch 2/10 | Loss: 0.5874 | Train Acc: 79.19%
Epoch 3/10 | Loss: 0.4429 | Train Acc: 85.23%
Epoch 4/10 | Loss: 0.3921 | Train Acc: 87.47%
Epoch 5/10 | Loss: 0.3300 | Train Acc: 90.38%
Epoch 6/10 | Loss: 0.2549 | Train Acc: 93.29%
Epoch 7/10 | Loss: 0.2432 | Train Acc: 92.62%
Epoch 8/10 | Loss: 0.2583 | Train Acc: 92.17%
Epoch 9/10 | Loss: 0.1934 | Train Acc: 95.08%
Epoch 10/10 | Loss: 0.1605 | Train Acc: 95.30%

Final Validation Accuracy: 98.67%

✅ Model and classes saved successfully to cnn_lstm_action_model.pth
***